Route feature engineering

In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = Path.cwd().parents[1]
DATA_FEB_DIR = PROJECT_ROOT / "Dataset_feb"
SAMSUNG_DIR = DATA_FEB_DIR / "Dataset_samsung"

input_file = SAMSUNG_DIR / "Routes.xlsx"
output_file = SAMSUNG_DIR / "rotas_features.xlsx"

print("Input file found:", input_file.exists())


# =========================
# LOAD ROUTE DATA
# =========================

df = pd.read_excel(input_file)


# =========================
# CONVERT DATE AND TIME COLUMNS
# =========================

df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])
df["day"] = pd.to_datetime(df["day"])


# =========================
# ALTITUDE FEATURES
# =========================

df["start_altitude"] = pd.to_numeric(
    df["start_altitude"],
    errors="coerce"
)

df["end_altitude"] = pd.to_numeric(
    df["end_altitude"],
    errors="coerce"
)

df["altitude_diff"] = (
    df["end_altitude"] - df["start_altitude"]
)

df["altitude_diff_abs"] = df["altitude_diff"].abs()


# =========================
# ROUTE DURATION
# =========================

df["route_duration_min"] = (
    (df["end_time"] - df["start_time"])
    .dt.total_seconds() / 60
)


# =========================
# ROUTE SLOPE TYPE
# =========================

def classify_route(x):
    if pd.isna(x):
        return "unknown"
    elif x > 5:
        return "uphill"
    elif x < -5:
        return "downhill"
    else:
        return "flat"


df["route_slope_type"] = (
    df["altitude_diff"]
    .apply(classify_route)
)


# =========================
# SPATIAL TRANSITION
# =========================

df["space_transition"] = (
    df["start_space_type"].astype(str)
    + "_to_"
    + df["end_space_type"].astype(str)
)


# =========================
# TIME OF DAY
# =========================

df["start_hour"] = df["start_time"].dt.hour


def period_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"


df["time_period"] = (
    df["start_hour"]
    .apply(period_of_day)
)


# =========================
# WEEKEND INDICATOR
# =========================

df["weekday"] = df["day"].dt.day_name()

df["is_weekend"] = (
    df["weekday"]
    .isin(["Saturday", "Sunday"])
)


# =========================
# GPS QUALITY
# =========================

def gps_quality(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 10:
        return "excellent"
    elif x <= 20:
        return "good"
    elif x <= 50:
        return "moderate"
    else:
        return "poor"


df["gps_quality"] = (
    df["avg_error"]
    .apply(gps_quality)
)


# =========================
# SAVE PROCESSED ROUTE FEATURES
# =========================

df.to_excel(
    output_file,
    index=False
)

print("Route feature file created successfully!")
print(f"Saved to: {output_file}")